PART 1

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, spark_partition_id

# 1. Initialize Spark
spark = SparkSession.builder.appName("Your Data? Our Data - Part 1").getOrCreate()

# Load the data first
pokemon_df = spark.read.csv("pokemon.csv", header=True, inferSchema=True)

print("Number of partitions (Default): ", pokemon_df.rdd.getNumPartitions())

# Partition 1 (Hash)
# Distributes data across 4 partitions based on the hash value of Type1 column
# Pokemon with the same Type1 will end up in the same partition
hash_df = pokemon_df.repartition(4, "Type1")
print("Number of partitions (Hash): ", hash_df.rdd.getNumPartitions())

# Show distribution of each partition
print("Hash Partition Distribution:")
hash_df.withColumn("PartitionID", spark_partition_id()).groupBy("PartitionID").count().orderBy("PartitionID").show()

# Partition 2 (Custom)
# Rule: Partition 0 = has evolution, Partition 1 = no evolution (final form)
pokemon_rdd = pokemon_df.rdd.map(lambda row: (0 if row["Evolution"] else 1, row))

custom_partitioned_rdd = pokemon_rdd.partitionBy(2)

print("Number of partitions (Custom): ", custom_partitioned_rdd.getNumPartitions())
print("Custom Partition Distribution:")
print("  Partition 0 (has evolution): ", custom_partitioned_rdd.filter(lambda x: x[0] == 0).count(), "rows")
print("  Partition 1 (no evolution): ", custom_partitioned_rdd.filter(lambda x: x[0] == 1).count(), "rows")

Number of partitions (Default):  1
Number of partitions (Hash):  4
Hash Partition Distribution:
+-----------+-----+
|PartitionID|count|
+-----------+-----+
|          0|  227|
|          1|  296|
|          2|  106|
|          3|  180|
+-----------+-----+

Number of partitions (Custom):  2
Custom Partition Distribution:
  Partition 0 (has evolution):  32 rows
  Partition 1 (no evolution):  777 rows


In [6]:
# Group pokemon by power type and count
specific_type = hash_df.groupBy("Type1").count()
print("Count of Pokemon per Type of Power: ")
# Display the summarized count table
specific_type.show()

Count of Pokemon per Type of Power: 
+--------+-----+
|   Type1|count|
+--------+-----+
|     Bug|   72|
| Psychic|   53|
|    Rock|   46|
|  Dragon|   27|
|    Dark|   29|
|   Water|  114|
|  Normal|  105|
|   Fairy|   18|
|Fighting|   29|
|   Ghost|   27|
|  Flying|    3|
|  Poison|   34|
|Electric|   40|
|  Ground|   32|
|   Grass|   78|
|    Fire|   53|
|     Ice|   23|
|   Steel|   26|
+--------+-----+



In [7]:
# Keep pokemons that have secondary power
dual_type = hash_df.filter(col("Type2") != "None")
print("Number of Dual-Type Pokemon: ")
# COunt the rows in the filtered dataset to print the number
print(dual_type.count())
# Choosing specific columns to display
dual_type.select("Name", "Type1", "Type2").show(20)

Number of Dual-Type Pokemon: 
405
+----------+-------+------+
|      Name|  Type1| Type2|
+----------+-------+------+
|butterfree|    Bug|Flying|
|    weedle|    Bug|Poison|
|    kakuna|    Bug|Poison|
|  beedrill|    Bug|Poison|
|     paras|    Bug| Grass|
|  parasect|    Bug| Grass|
|   venonat|    Bug|Poison|
|  venomoth|    Bug|Poison|
|   geodude|   Rock|Ground|
|  graveler|   Rock|Ground|
|     golem|   Rock|Ground|
|      onix|   Rock|Ground|
|   mr-mime|Psychic| Fairy|
|   scyther|    Bug|Flying|
|   omanyte|   Rock| Water|
|   omastar|   Rock| Water|
|    kabuto|   Rock| Water|
|  kabutops|   Rock| Water|
|aerodactyl|   Rock|Flying|
| dragonite| Dragon|Flying|
+----------+-------+------+
only showing top 20 rows


In [8]:
# SORTED BY NAME (A-Z)

# 1. Sort the hash-partitioned dataset by Name alphabetically
alphabetical_df = hash_df.orderBy("Name")
print("Total Pokemon in alphabetical list: ")

# Display the total count of the dataset
print(alphabetical_df.count())

# Use count() dynamically to ensure every single row is displayed in the table
alphabetical_df.show(alphabetical_df.count(), truncate=False)

Total Pokemon in alphabetical list: 
809
+--------------------+--------+--------+----------+
|Name                |Type1   |Type2   |Evolution |
+--------------------+--------+--------+----------+
|abomasnow           |Grass   |Ice     |NULL      |
|abra                |Psychic |NULL    |kadabra   |
|absol               |Dark    |NULL    |NULL      |
|accelgor            |Bug     |NULL    |NULL      |
|aegislash-blade     |Steel   |Ghost   |NULL      |
|aerodactyl          |Rock    |Flying  |NULL      |
|aggron              |Steel   |Rock    |NULL      |
|aipom               |Normal  |NULL    |NULL      |
|alakazam            |Psychic |NULL    |NULL      |
|alomomola           |Water   |NULL    |NULL      |
|altaria             |Dragon  |Flying  |NULL      |
|amaura              |Rock    |Ice     |NULL      |
|ambipom             |Normal  |NULL    |NULL      |
|amoonguss           |Grass   |Poison  |NULL      |
|ampharos            |Electric|NULL    |NULL      |
|anorith             |R

In [9]:
# TRANSFORMATIONS FOR CUSTOM PARTITION

has_evolution = custom_partitioned_rdd.filter(lambda x: x[0] == 0).map(lambda x: x[1]).toDF(pokemon_df.schema)

# We use the 'has_evolution' dataframe created during partitioning
# This focuses only on Partition 0
evolving_list = has_evolution.select("Name", "Evolution")

print("Total Pokemon capable of evolving: ")
# This count comes specifically from the 'has_evolution' partition
print(evolving_list.count())

# We show all rows in this partition
evolving_list.show(evolving_list.count(), truncate=False)

Total Pokemon capable of evolving: 
32
+----------+----------+
|Name      |Evolution |
+----------+----------+
|bulbasaur |ivysaur   |
|ivysaur   |venusaur  |
|charmander|charmeleon|
|charmeleon|charizard |
|squirtle  |wartortle |
|wartortle |blastoise |
|caterpie  |metapod   |
|metapod   |butterfree|
|weedle    |kakuna    |
|kakuna    |beedrill  |
|pidgey    |pidgeotto |
|pidgeotto |pidgeot   |
|rattata   |raticate  |
|spearow   |fearow    |
|ekans     |arbok     |
|pikachu   |raichu    |
|sandshrew |sandslash |
|nidoran-f |nidorina  |
|nidorina  |nidoqueen |
|nidoran-m |nidorino  |
|nidorino  |nidoking  |
|clefairy  |clefable  |
|vulpix    |ninetales |
|diglett   |dugtrio   |
|meowth    |persian   |
|psyduck   |golduck   |
|mankey    |primeape  |
|growlithe |arcanine  |
|poliwag   |poliwhirl |
|poliwhirl |poliwrath |
|abra      |kadabra   |
|kadabra   |alakazam  |
+----------+----------+

